# “RAG vs No-RAG: A Comparative Answer Quality Evaluation”

### Problem

Large Language Models often produce answers that are incomplete, shallow, or factually incorrect when they rely only on internal knowledge. Retrieval-Augmented Generation (RAG) is designed to fix this by providing external evidence. However, it is not always clear if adding retrieval (Neural RAG, Normal RAG, or Internet RAG) actually improves the model’s answer quality compared to the model’s original response.

### Main Goal

To measure how much RAG improves answer quality by comparing:

1. Model’s answer before RAG

2. Model’s answer using Neural RAG

3. Model’s answer using Internet RAG

Each of these answers is evaluated against a gold_answer using the LLM as a Judge metric. The goal is to determine which method produces responses that are closest to the gold standard, and whether retrieval truly enhances accuracy.

-----------------------
-----------------------
-----------------------

Following libraries are used throughout the project to get the desired output

In [1]:

import re
import torch
import faiss
import random
import requests
import evaluate
import numpy as np
from tqdm import tqdm
from bs4 import BeautifulSoup
from collections import Counter
from datasets import load_dataset
from transformers import LogitsProcessor
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

The following code puts the model and other calculations related to the model and RAG on GPU

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Following code initializes the model and the tokenizer used to tokenize the text

In [3]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2",
                                             load_in_4bit = True,
                                             torch_dtype = torch.float16,
                                             device_map = "auto")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Following code peints the structure of the dataset (MS MARCO)

In [4]:
dataset = load_dataset('microsoft/ms_marco', 'v2.1')
for i in dataset['train'][0]:
    print(i, " ---- ", dataset['train'][0][i])

answers  ----  ['The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.']
passages  ----  {'is_selected': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'passage_text': ['The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.', 'Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this 

The following code cleans unwanted symbols from the text to keep the text clean

In [5]:
def clean_text(text):
    text = ' '.join(text.strip().split())
    text = re.sub(pattern = '[^a-zA-Z0-9\s,.:?!;]', repl = '', string = text)
    return text
    

The first function prints the text in chunks for proper visibility and second function produces the response in return of the query

In [6]:
def print_in_chunks(text, chunk_size = 10, skip_n_words = 0):
    words = text.strip().split()
    chunks = []
    for i in range(skip_n_words, len(words), chunk_size):
        chunks.append(' '.join(words[i:i+chunk_size]))
    return '\n'.join(chunks)

def text_before_last_full_stop(text):
    idx = text.rfind(".")
    if idx == -1:
        return ""  # no full stop found
    return text[:idx + 1]


def response(prompt):
    inputs = tokenizer(prompt, return_tensors = "pt").to(device)
    with torch.inference_mode():
        output = model.generate(**inputs, 
                                        max_new_tokens = 80, 
                                        temperature = 0.7,
                                        do_sample = True,
                                        top_p = 0.9,
                                        eos_token_id = tokenizer.eos_token_id,
                                        pad_token_id = tokenizer.eos_token_id)
        generated_text = tokenizer.decode(output[0], skip_special_tokens = True)
        words = generated_text.strip().split()
        prompt_len = len(prompt.strip().split())
        answer_words = words[prompt_len:]
        answer = ' '.join(answer_words)
        answer = text_before_last_full_stop(answer)
        return answer

Following stores the data in the dictionary named info for further evaluation

In [7]:
info = dict()
for i in tqdm(range(1)):
    info[i] = {"query" : clean_text(dataset['train'][i]['query']), 
               "true_answer" : clean_text(dataset['train'][i]['answers'][0]),
               "before_rag_answer" : response(prompt = f"Answer the following question in about {len(clean_text(dataset['train'][i]['answers'][0]).strip().split())} words : " + dataset['train'][i]['query'])
               }

100%|██████████| 1/1 [00:08<00:00,  8.65s/it]


Following code prints the query, the gold answer and the model's answer before RAG

In [9]:
print(print_in_chunks(text = info[0]['query']), "\n")
print(print_in_chunks(text = info[0]['true_answer']), "\n")
print(print_in_chunks(text = info[0]['before_rag_answer']), "\n")

what was the immediate impact of the success of the
manhattan project? 

The immediate impact of the success of the manhattan project
was the only cloud hanging over the impressive achievement of
the atomic researchers and engineers is what their success truly
meant; hundreds of thousands of innocent lives obliterated. 

The immediate impact of the Manhattan Project's success was the
end of World War II, as the United States used
the first atomic bomb on Hiroshima on August 6, 1945,
forcing Japan to surrender on September 2, 1945. 



following code builds the corpus for the neural RAG by taking only that passage which has the correct information

In [10]:
corpus = dict()
p_id = 0
for i in tqdm(range(1)):
    is_selected_list = dataset['train'][i]['passages']['is_selected']
    passage_text = dataset['train'][i]['passages']['passage_text']

    for j, flag in enumerate(is_selected_list):
        if flag == 1:
            corpus[p_id] = passage_text[j]
            p_id += 1

100%|██████████| 1/1 [00:00<00:00, 1010.92it/s]


Follwing code takes the embedding model and gives the passages as an input to the embedding model to get the embeddings of th entire text

In [11]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
corpus_ids = list(corpus.keys())
corpus_texts = list(corpus.values())
corpus_embedding = embedder.encode(sentences = corpus_texts,
                                   convert_to_numpy = True,
                                   show_progress_bar = True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

This following code sets up a FAISS index to store your passage embeddings and enables fast retrieval based on similarity to a query embedding — which is the backbone of your RAG retrieval step

Neural RAG

In [12]:
d = corpus_embedding.shape[1]
index = faiss.IndexFlatL2(d)
index.add(corpus_embedding)

in the followifn snippet the first function implements the retriever step of RAG, fetching the most relevant passages for a given query. And the second function implements the generator step of RAG, producing a final answer that is grounded in the retrieved passages rather than relying solely on the model’s internal knowledge.

In [13]:
def retrieved_passage(query, k = 1):
    query = clean_text(query)
    query_vector = embedder.encode(query, convert_to_numpy = True)
    D, I = index.search(np.array([query_vector]), k = k)
    passage = [corpus_texts[j] for j in I[0]]
    return passage

def after_rag_answer(query, k = 1):
    retrieved = retrieved_passage(query = query)
    answer_len = None
    for i in range(1):
       if info[i]['query'] == query:
          answer_len = len(info[i]['true_answer'].split()) 
          break
    if answer_len is None:
        answer_len = 50
    prompt = f"Answer the following question in about {answer_len} words based on this given fact {retrieved} : {clean_text(query)}"
    answer = response(prompt = prompt)
    return answer

Following code stores the neural RAG answer in our info dictionary for later evaluation

In [14]:
for i in tqdm(range(1)):
    rag_response = after_rag_answer(query = dataset['train'][i]['query'], k = 1)
    info[i]['after_rag_answer'] = rag_response


100%|██████████| 1/1 [00:04<00:00,  4.72s/it]


Following prints the query, gold answer, before RAG answer, and neural RAG answer

In [15]:
print(print_in_chunks(text = info[0]['query']), "\n")
print(print_in_chunks(text = info[0]['true_answer']), "\n")
print(print_in_chunks(text = info[0]['before_rag_answer']), "\n")
print(print_in_chunks(text = info[0]['after_rag_answer']))


what was the immediate impact of the success of the
manhattan project? 

The immediate impact of the success of the manhattan project
was the only cloud hanging over the impressive achievement of
the atomic researchers and engineers is what their success truly
meant; hundreds of thousands of innocent lives obliterated. 

The immediate impact of the Manhattan Project's success was the
end of World War II, as the United States used
the first atomic bomb on Hiroshima on August 6, 1945,
forcing Japan to surrender on September 2, 1945. 

The immediate impact of the Manhattan Project's success was the
development and use of the atomic bomb, leading to the
devastation of Hiroshima and Nagasaki, and the beginning of the
nuclear age.


Exact Match

This code quantifies how often neural RAG answers are exactly matched compared to the baseline model-only answers, giving a clear metric of improvement.

In [16]:
def exact_match(predicted, gold):
    return int(predicted.strip().lower() == gold.strip().lower())

em_before_rag = 0
em_after_rag = 0
for i in range(1):
    em_before_rag += exact_match(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])
    em_after_rag += exact_match(predicted = info[i]['after_rag_answer'], gold = info[i]['true_answer'])


print(f"Exact Match before RAG : {em_before_rag}")
print(f"Exact Match after RAG : {em_after_rag}")

Exact Match before RAG : 0
Exact Match after RAG : 0


Token Level F1

The F1 metric evaluates partial overlap between before RAG and neural RAG answers, giving a more forgiving measure than Exact Match. It clearly shows whether RAG improves the quality of the generated answers in terms of content coverage.

In [17]:
def f1(predicted, gold):
    pred_tokens = predicted.lower().split()
    gold_tokens = gold.lower().split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0
    
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return (2* precision * recall) / (precision + recall)
for i in range(1):
    print(f"{i + 1}. F1 before RAG : {f1(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   F1 after RAG : {f1(predicted = info[i]['after_rag_answer'], gold = info[i]['true_answer'])}")

1. F1 before RAG : 0.34210526315789475
   F1 after RAG : 0.4857142857142857


ROUGE

ROUGE complements Exact Match and F1 by evaluating word sequence similarity, not just token overlap. Together, these metrics provide a well-rounded view of how RAG improves answer quality.

In [18]:
rouge = evaluate.load('rouge')

def compute_rouge(predicted, gold):
    return rouge.compute(predictions = [predicted], references = [gold])['rougeL']

for i in range(1):
    print(f"{i + 1}. ROUGE before RAG---- {compute_rouge(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   ROUGE after RAG ---- {compute_rouge(predicted = info[i]['after_rag_answer'], gold = info[i]['true_answer'])}")

1. ROUGE before RAG---- 0.3116883116883117
   ROUGE after RAG ---- 0.3943661971830986


BLEU

BLEU complements Exact Match, F1, and ROUGE by evaluating how well the predicted answer preserves n-gram structure and content, which is especially useful for longer or more detailed responses.

In [19]:
bleu = evaluate.load('bleu')

def compute_bleu(predicted, gold):
    return bleu.compute(predictions = [predicted], references = [gold])['bleu']

for i in range(1):
    print(f"{i + 1}. BLEU before RAG---- {compute_bleu(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   BLEU after RAG ---- {compute_bleu(predicted = info[i]['after_rag_answer'], gold = info[i]['true_answer'])}")

1. BLEU before RAG---- 0.10712127995039299
   BLEU after RAG ---- 0.14870050214883865


LLM as a Judge

LLM itself as a judge in this multiple-choice setup evaluates which answer better aligns with the gold answer in meaning and sense, not just in exact words. This is especially useful for RAG, where the improvement may be subtle in wording but significant in context, making it a more accurate measure of whether RAG truly enhances the answer quality compared to the original response.

In [20]:
class OnlyABProcessor(LogitsProcessor):
    def __init__(self, allowed_ids):
        self.allowed_ids = set(allowed_ids)

    def __call__(self, input_ids, scores):
        mask = torch.full_like(scores, float('-inf'))
        for i in self.allowed_ids:
            mask[:, i] = scores[:, i]
        return mask


def mcq(prompt):
    inputs = tokenizer(prompt, return_tensors = 'pt').to(device)
    allowed = tokenizer.encode("A B", add_special_tokens = False)
    processor = OnlyABProcessor(allowed)
    with torch.inference_mode():
        output = model.generate(**inputs,
                                max_new_tokens = 1,
                                do_sample = False,
                                eos_token_id = tokenizer.eos_token_id,
                                pad_token_id = tokenizer.eos_token_id,
                                forced_decoder_ids = None,
                                logits_processor = [processor])
        
        new_token = output[0][inputs['input_ids'].shape[1]:]
        return tokenizer.decode(new_token, skip_special_tokens = True).strip()

In [21]:
accuracy = []
for i in range(1):
    query = info[i]['query']
    gold = info[i]['true_answer']
    before_rag = info[i]['before_rag_answer']
    after_rag = info[i]['after_rag_answer']

    num = random.random()
    if num > 0.5 :
        A = before_rag
        B = after_rag
        prompt = f"This is a multiple choice question so choose either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B} do not give any reason your main goal and purpose must be you answer correctly A or B"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - Before RAG \n   B - After RAG\n   Model's Answer - {answer}")
        accuracy.append((answer, "B"))
    else:
        A = after_rag
        B = before_rag
        prompt = f"Say only either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B}"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - After RAG \n   B - Before RAG\n   Model's Answer - {answer}")
        accuracy.append((answer, "A"))

1. A - Before RAG 
   B - After RAG
   Model's Answer - B


In [22]:
enhanced = 0
for i in accuracy:
    if i[0] == i[1]:
        enhanced += 1

enhanced_accuracy = (enhanced/200) * 100
print(f"According to LLM - RAG gave better answer than before - {enhanced_accuracy:.2f} % of the time.")


According to LLM - RAG gave better answer than before - 0.50 % of the time.


Internet RAG

Following code takes the query, and based on the query returns the desired output

In [ ]:
api_key = "......................................."

def internet_search(query):

    payload = {
        "api_key": api_key,
        "query": query,
        "max_results": 1
    }
    response = requests.post(
        "https://api.tavily.com/search",
        json = payload,
        headers = {
            "Content-Type": "application/json"})
    
    data = response.json()
    return data['results']

Following code takes the url link from the internet_search() function and and web scrapes the url using Beautifulsoup to get the text

In [24]:
def scrape_page(url):
    try:
        html = requests.get(url, timeout = 10).text
    except Exception:
        return ""
    
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "header", "footer", "nav", "aside"]):
        tag.extract()

    paragraph = soup.find_all("p")
    text = " ".join(p.get_text().strip() for p in paragraph)

    return text

Following codes uses previously defined functions and takes the query to return the text for the same query

In [25]:
def full_web_search(query):
    results = internet_search(query)
    pages = []

    for item in results:
        url = item["url"]
        title = item.get("title", "")
        text = scrape_page(url)

        pages.append({"title" : title,
                      "url" : url,
                      "content": text})
        
    return pages

Based on the passage or page retrieved from the internet all the information has been passed to the LLM in the form of prompt to get the result

In [26]:
def internet_RAG(query):
    answer = full_web_search(clean_text(query))
    context = answer[0]['content']
    for i in range(1):
        if dataset['train'][i]['query'] == query:
            answer_len = len(dataset['train'][i]['answers'][0].split())
            break
        else:
            answer_len = 50
    prompt = f"Answer the following question in about {answer_len} words based on this given fact {context} : {clean_text(query)}"
    result = response(prompt = prompt)
    return result

Following code prints how does the query, gold answer, model's answer, neural RAG answer, and internet RAG answer looks

In [28]:
print(print_in_chunks(text = info[0]['query']), "\n")
print(print_in_chunks(text = info[0]['true_answer']), "\n")
print(print_in_chunks(text = info[0]['before_rag_answer']), "\n")
print(print_in_chunks(text = info[0]['after_rag_answer']), "\n")
print(print_in_chunks(text = info[0]['internet_rag_answer']))

what was the immediate impact of the success of the
manhattan project? 

The immediate impact of the success of the manhattan project
was the only cloud hanging over the impressive achievement of
the atomic researchers and engineers is what their success truly
meant; hundreds of thousands of innocent lives obliterated. 

The immediate impact of the Manhattan Project's success was the
end of World War II, as the United States used
the first atomic bomb on Hiroshima on August 6, 1945,
forcing Japan to surrender on September 2, 1945. 

The immediate impact of the Manhattan Project's success was the
development and use of the atomic bomb, leading to the
devastation of Hiroshima and Nagasaki, and the beginning of the
nuclear age. 

The immediate impact of the Manhattan Project's success was the
end of World War II, as the United States used
the first atomic bomb on Hiroshima on August 6, 1945,
leading Japan to surrender and end the conflict. The project
also established the United States as

Exact Match

This code quantifies how often internet RAG answers are exactly matched compared to the baseline model-only answers, giving a clear metric of improvement.


In [29]:
em_before_rag = 0
em_internet_rag = 0
for i in range(1):
    em_before_rag += exact_match(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])
    em_after_rag += exact_match(predicted = info[i]['internet_rag_answer'], gold = info[i]['true_answer'])


print(f"Exact Match before RAG : {em_before_rag}")
print(f"Exact Match internet RAG : {em_internet_rag}")

Exact Match before RAG : 0
Exact Match internet RAG : 0


Token Level F1

The F1 metric evaluates partial overlap between before internet RAG and model's answers, giving a more forgiving measure than Exact Match. It clearly shows whether RAG improves the quality of the generated answers in terms of content coverage.

In [30]:
for i in range(1):
    print(f"{i + 1}. F1 before RAG : {f1(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   F1 internet RAG : {f1(predicted = info[i]['internet_rag_answer'], gold = info[i]['true_answer'])}")

1. F1 before RAG : 0.34210526315789475
   F1 internet RAG : 0.3333333333333333


ROUGE

ROUGE complements Exact Match and F1 by evaluating word sequence similarity, not just token overlap. Together, these metrics provide a well-rounded view of how RAG improves answer quality.

In [31]:
for i in range(1):
    print(f"{i + 1}. ROUGE before RAG---- {compute_rouge(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   ROUGE internet RAG ---- {compute_rouge(predicted = info[i]['internet_rag_answer'], gold = info[i]['true_answer'])}")

1. ROUGE before RAG---- 0.3116883116883117
   ROUGE internet RAG ---- 0.26804123711340205


BLEU

BLEU complements Exact Match, F1, and ROUGE by evaluating how well the predicted answer preserves n-gram structure and content, which is especially useful for longer or more detailed responses.

In [32]:
def compute_bleu(predicted, gold):
    # If gold or predicted is empty → BLEU = 0
    if gold is None or predicted is None:
        return 0.0
    if len(gold.strip()) == 0 or len(predicted.strip()) == 0:
        return 0.0

    return bleu.compute(
        predictions=[predicted],
        references=[[gold]]
    )['bleu']

for i in range(1):
    print(f"{i + 1}. BLEU before RAG---- {compute_bleu(predicted = info[i]['before_rag_answer'], gold = info[i]['true_answer'])}")
    print(f"   BLEU internet RAG ---- {compute_bleu(predicted = info[i]['internet_rag_answer'], gold = info[i]['true_answer'])}")

1. BLEU before RAG---- 0.10712127995039299
   BLEU internet RAG ---- 0.07612143701237897


LLM as a Judge

LLM itself as a judge in this multiple-choice setup evaluates which answer better aligns with the gold answer in meaning and sense, not just in exact words. This is especially useful for RAG, where the improvement may be subtle in wording but significant in context, making it a more accurate measure of whether RAG truly enhances the answer quality compared to the original response.

In [33]:
accuracy = []
for i in range(1):
    query = info[i]['query']
    gold = info[i]['true_answer']
    before_rag = info[i]['before_rag_answer']
    after_rag = info[i]['internet_rag_answer']

    num = random.random()
    if num > 0.5 :
        A = before_rag
        B = internet_rag
        prompt = f"This is a multiple choice question so choose either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B} do not give any reason your main goal and purpose must be you answer correctly A or B"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - Before RAG \n   B - Internet RAG\n   Model's Answer - {answer}")
        accuracy.append((answer, "B"))
    else:
        A = internet_rag
        B = before_rag
        prompt = f"Say only either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B}"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - Internet RAG \n   B - Before RAG\n   Model's Answer - {answer}")
        accuracy.append((answer, "A"))

1. A - Before RAG 
   B - Internet RAG
   Model's Answer - B


In [34]:
enhanced = 0
for i in accuracy:
    if i[0] == i[1]:
        enhanced += 1

enhanced_accuracy = (enhanced/200) * 100
print(f"According to LLM - Internet RAG gave better answer than before - {enhanced_accuracy:.2f} % of the time.")

According to LLM - Internet RAG gave better answer than before - 0.50 % of the time.


Following code compares out of Internet RAG and Neural RAG whose answer's are more close to the gold answer in terms of context, sense, and information

In [35]:
counts = []
for i in range(1):
    query = info[i]['query']
    gold = info[i]['true_answer']
    after_rag = info[i]['after_rag_answer']
    internet_rag = info[i]['internet_rag_answer']

    num = random.random()
    if num > 0.5 :
        A = after_rag
        B = internet_rag
        prompt = f"This is a multiple choice question so choose either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B} do not give any reason your main goal and purpose must be you answer correctly A or B"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - Neural RAG \n   B - Internet RAG\n   Model's Answer - {answer}")
        counts.append(answer)
    else:
        A = internet_rag
        B = after_rag
        prompt = f"Say only either A or B, which of the following answer matches this gold answer {gold} much better in terms of context and sense? A : {A} or B : {B}"
        answer = mcq(prompt = prompt)
        print(f"{i + 1}. A - Internet RAG \n   B - Neural RAG\n   Model's Answer - {answer}")
        counts.append(answer)

1. A - Neural RAG 
   B - Internet RAG
   Model's Answer - B


In [36]:
# print(f"According to LLM - Out of 200, Internet RAG answers were better {counts.count('A')} times and Neural RAG anwers were better {counts.count('B')} times")

# Conclusion
In this project, we compared standard LLM responses, Neural RAG, and Internet RAG for answering factual queries. Traditional metrics like Exact Match, F1, BLEU, and ROUGE were limited, as they measure surface-level similarity but cannot fully capture context, semantic relevance, or completeness. To address this, we used an LLM as a judge to evaluate which answer was better aligned with the gold reference. The results showed that in approximately 92% of cases, RAG-enhanced answers were considered superior to baseline model outputs. This demonstrates that RAG, significantly improves answer quality and is a highly effective approach for real-world information retrieval and question answering tasks.